# Reconciling two health facility lists

**What this notebook does:** takes two independent lists of Nigerian health
facilities, works out which records refer to the same real clinic, and — just
as importantly — refuses to guess when the evidence doesn't support a verdict.

**Who it's for:** anyone new to entity resolution. No prior knowledge assumed.
Every step prints what it did.

**Runtime:** about a minute. Everything runs offline on public data already in
this repository. No API key, no network call, no model download.

---

## The problem, in one paragraph

A country's health system runs on a **Master Facility List**: the canonical
answer to *where are our clinics, and which record is which?* It decides where
vaccines ship and how coverage is measured.

The lists don't line up. The government registry spells a clinic one way,
OpenStreetMap another, and the coordinates disagree because one team measured
at the gate and the other at the junction. Reconciling them by hand is real
work — a peer-reviewed effort in Senegal needed *three rounds of manual expert
verification* to merge 16 datasets into one list.

## The two lists we'll use

| List | What it is | Why it's a fair test |
|---|---|---|
| **GRID3** | Gates/FCDO-funded geospatial reference data | The candidate master list |
| **OpenStreetMap** | Crowd-mapped, via the Overpass API | **Independent** — not derived from the government registry |

That independence matters. Nigeria's official registry (NHFR) and GRID3 share
data — 68% of GRID3 facility names carry `facility_name_source: NHFR_2024`.
Comparing those two would mostly measure agreement between two views of the
same ancestor. OpenStreetMap is genuinely separate, so matching against it is a
real reconciliation rather than a pre-merged softball.

## Step 1 — Load the two lists

Plain `csv`, no dependencies.

In [1]:
import csv, collections, warnings
warnings.filterwarnings("ignore")

STATE = "Kano"

with open("../../data/GRID3_NGA_health_facilities_v2.csv", encoding="utf-8-sig") as fh:
    grid3 = [r for r in csv.DictReader(fh) if r["state"] == STATE]

with open("../../data/osm_kano.csv", encoding="utf-8-sig") as fh:
    osm = [r for r in csv.DictReader(fh) if r.get("name", "").strip()]

print(f"GRID3 {STATE}: {len(grid3):,} facilities")
print(f"OSM   {STATE}: {len(osm):,} named facilities")
print()
print("A GRID3 record looks like:")
for k in ("facility_name", "lga", "ward", "facility_level_option", "latitude", "longitude"):
    print(f"   {k:24} {grid3[0][k]}")
print()
print("An OSM record looks like:")
for k, v in osm[0].items():
    print(f"   {k:24} {v}")

GRID3 Kano: 1,723 facilities
OSM   Kano: 685 named facilities

A GRID3 record looks like:
   facility_name            Yadakunya Leprosy General Hospital
   lga                      Ungogo
   ward                     Yadakunya
   facility_level_option    General Hospital
   latitude                 12.0839139997
   longitude                8.63161999998

An OSM record looks like:
   id                       node/2923098159
   name                     Tsara Primary Health Centre
   amenity                  hospital
   healthcare               
   lga                      Rogo
   lat                      11.4507448
   lon                      7.7276659


## Step 2 — Why you can't just compare the strings

The obvious approach is to lowercase both names and check equality. Let's see
how far that gets us, because the answer sets up everything that follows.

In [2]:
def norm(s):
    return " ".join(s.strip().lower().split())

grid3_names = {norm(r["facility_name"]) for r in grid3}
exact = [r for r in osm if norm(r["name"]) in grid3_names]
residue = [r for r in osm if norm(r["name"]) not in grid3_names]

print(f"OSM records                    : {len(osm):,}")
print(f"  exact name match in GRID3    : {len(exact):,}  ({100*len(exact)/len(osm):.1f}%)")
print(f"  no exact match               : {len(residue):,}  ({100*len(residue)/len(osm):.1f}%)")
print()
print("Ten names with no exact match — look at WHY they fail:")
for r in residue[:10]:
    print("   ", r["name"])

OSM records                    : 685
  exact name match in GRID3    : 309  (45.1%)
  no exact match               : 376  (54.9%)

Ten names with no exact match — look at WHY they fail:
    Tsara Primary Health Centre
    Fulatan Primary Health Centre
    Gwangwan Primary Health Centre
    Hago Primary Health Centre
    Unguwar Malam Amadu Health Post
    Zarewa Primary Health Centre
    Bari Primary Health Centre
    Karshi Health Post
    Tsohuwar Rogo Health Post
    Ba'Awa Health Post


**Read that list carefully.** These aren't exotic cases. They're the same
clinic written slightly differently: a `Centre`/`Center` spelling, a dropped
qualifier, a word order swap, an apostrophe.

This is the honest shape of the problem:

- **The exact matches need no product at all.** A `dict` lookup solves them.
- **The residue is the entire job**, and it splits again into pairs you can
  safely merge and pairs where merging would be a mistake.

Anyone selling you a platform for the first group is selling you a dictionary.

## Step 3 — Two ways string matching fails

Before reaching for a tool, it's worth seeing both failure modes, because they
pull in opposite directions and that's what makes the problem hard.

In [3]:
from rapidfuzz import fuzz

pairs = [
    ("An Nur Specialist Hospital",  "Al Noury Specialist Hospital",  "same place, two transliterations"),
    ("Central Dispensary",          "Central Dispensary",            "identical name, 3 km apart"),
    ("Kirya Health Post",           "Zaura Health Post",             "different places, shared type"),
    ("Tsalle Health Post",          "Tsalle Primary Health Care Center", "same place, different tier?"),
]
print(f"{'A':36} {'B':38} {'token_sort':>10}  note")
print("-" * 108)
for a, b, note in pairs:
    print(f"{a:36} {b:38} {fuzz.token_sort_ratio(a, b):>10.0f}  {note}")

A                                    B                                      token_sort  note
------------------------------------------------------------------------------------------------------------
An Nur Specialist Hospital           Al Noury Specialist Hospital                   93  same place, two transliterations
Central Dispensary                   Central Dispensary                            100  identical name, 3 km apart
Kirya Health Post                    Zaura Health Post                              65  different places, shared type
Tsalle Health Post                   Tsalle Primary Health Care Center              59  same place, different tier?


Two things go wrong at once:

**It over-merges.** Two `Central Dispensary` records score 100 even when
they're three kilometres apart and are plainly different clinics.

**It under-matches.** `An Nur` and `Al Noury` are the same Arabic name
transliterated two ways, but character-by-character they barely overlap.

And a third problem hides underneath: `Health Post` appears in half the names,
so the shared *type* token inflates every score while the **distinctive** part
— the place name — gets drowned out.

A single similarity threshold cannot fix this. Raise it and you lose the
transliterations; lower it and you merge distinct clinics. You need the
comparison to know that `Health Post` is a type and `Tsalle` is a name.

## Step 4 — Run arche

`crosswalk` takes two lists of records and returns the edges between them.
`entity="place"` selects the place comparator: type tokens get stripped to
canonical form, geography is a *supporting* signal rather than a decider, and
names are compared with string similarity rather than the person-name
equivalence lexicon.

In [4]:
from arche.resolve import crosswalk
import time

A = [{"name": r["name"], "lat": r["lat"], "lon": r["lon"], "type": r.get("amenity", "")}
     for r in osm]
B = [{"name": r["facility_name"], "lat": r["latitude"], "lon": r["longitude"],
      "type": r["facility_level_option"]} for r in grid3]

t0 = time.time()
result = crosswalk(A, B, entity="place")
elapsed = time.time() - t0

print(f"compared      : {len(A):,} OSM x {len(B):,} GRID3 = {len(A)*len(B):,} possible pairs")
print(f"blocked down  : {result['blocking']['candidate_pairs']:,} pairs actually scored")
print(f"reduction     : {result['blocking']['reduction_ratio']:.2%}")
print(f"edges returned: {result['count']:,}")
print(f"wall clock    : {elapsed:.1f}s")

compared      : 685 OSM x 1,723 GRID3 = 1,180,255 possible pairs
blocked down  : 39,701 pairs actually scored
reduction     : 96.64%
edges returned: 907
wall clock    : 20.9s


### What just happened

**Blocking** is why this is fast. Comparing every OSM record against every
GRID3 record would be over a million comparisons. arche first generates
*candidate* pairs using cheap signals, then scores only those.

Crucially it does **not** block on geography alone. It ORs three strategies:

- `h3` — facilities in the same spatial cell
- `token` — facilities sharing a *rare* word (`Tsalle` is rare; `Health` is not)
- `id` — facilities sharing an identifier

That matters because coordinates in this data are often wrong by kilometres.
Blocking on geography alone silently drops true matches, and a dropped pair can
never be recovered downstream.

## Step 5 — Read the verdicts

Every edge carries one of two decisions. There is no third "probably".

In [5]:
decisions = collections.Counter(m["decision"] for m in result["matches"])
print("edges by decision:", dict(decisions))
print()

matched   = {m["a_id"] for m in result["matches"] if m["decision"] == "match"}
reviewed  = {m["a_id"] for m in result["matches"] if m["decision"] == "review"} - matched
unmatched = len(A) - len(matched) - len(reviewed)

print(f"OSM records resolved to GRID3 : {len(matched):>4}  ({100*len(matched)/len(A):.1f}%)")
print(f"OSM records sent to review    : {len(reviewed):>4}  ({100*len(reviewed)/len(A):.1f}%)")
print(f"OSM records with no candidate : {unmatched:>4}  ({100*unmatched/len(A):.1f}%)")

edges by decision: {'match': 618, 'review': 289}

OSM records resolved to GRID3 :  532  (77.7%)
OSM records sent to review    :  100  (14.6%)
OSM records with no candidate :   53  (7.7%)


## Step 6 — The matches

These are pairs arche was willing to merge. Note what it saw through.

In [6]:
M = sorted([m for m in result["matches"] if m["decision"] == "match"],
           key=lambda m: -m["score"])

print(f"{'score':>6} {'km':>6}  {'OSM':38} GRID3")
print("-" * 100)
for m in M[:12]:
    d = m["evidence"].get("distance_km", 0)
    print(f'{m["score"]:>6.3f} {d:>6.2f}  {A[m["a_id"]]["name"][:36]:38} {B[m["b_id"]]["name"][:36]}')

 score     km  OSM                                    GRID3
----------------------------------------------------------------------------------------------------
 1.000   0.00  Bargoni Health Post                    Bargoni Health Post
 1.000   0.00  Baskore Health Post                    Baskore Health Post
 1.000   0.00  Husama Health Post                     Husama Health Post
 1.000   0.00  Walawa Health Post                     Walawa Health Post
 1.000   0.00  Gurawa Health Post                     Gurawa Health Post
 1.000   0.00  Maraku Health Post                     Maraku Health Post
 1.000   0.00  Bankaura Health Post                   Bankaura Health Post
 1.000   0.00  Tofawa Health Post                     Tofawa Health Post
 1.000   0.00  Farsa Health Post                      Farsa Health Post
 1.000   0.00  Gidan Nisau Health Post                Gidan Nisau Health Post
 1.000   0.00  Mazan Gudu Health Post                 Mazan Gudu Health Post
 1.000   0.00  Kwarkiya 

## Step 7 — The review queue (the important part)

These pairs scored well and were **still refused**. This is the product.

Read each one and ask yourself whether you'd have merged it.

In [7]:
R = sorted([m for m in result["matches"] if m["decision"] == "review"],
           key=lambda m: -m["score"])

print(f"{'score':>6} {'km':>6}  {'OSM':38} GRID3")
print("-" * 100)
for m in R[:12]:
    d = m["evidence"].get("distance_km", 0)
    print(f'{m["score"]:>6.3f} {d:>6.2f}  {A[m["a_id"]]["name"][:36]:38} {B[m["b_id"]]["name"][:36]}')

 score     km  OSM                                    GRID3
----------------------------------------------------------------------------------------------------
 0.698   0.38  Gurduba Health Post                    Gurduba Primary Health Care
 0.695   0.04  Lambu Primary Health Centre            Lambu Basic Health Center
 0.693   0.02  Dawaki General Hospital                Dawakin Kudu General Hospital
 0.691   0.00  Tsalle Health Post                     Tsalle Primary Health Care Center
 0.689   0.99  Tariwa Primary Health Centre           Tariwa Health Post
 0.686   2.64  Jibga Fulani Health Post               Jibga Health Post
 0.684   0.14  Sauna Kawaji Health Post               Sauna Primary Health Center
 0.683   0.13  National Orthopaedic Hospital          Dala National Orthophedic Hospital
 0.680   0.05  Dawaki General Hospital                Dawakin Tofa General Hospital
 0.675   0.05  Dadin Kowa Primary Health Centre       Dadin Kowa Nursing and Maternity Hom
 0.672   0.02 

Look at what's in that queue:

- **`Gurduba Health Post` vs `Gurduba Primary Health Care`** — same settlement,
  but a health post and a primary health care centre are different tiers of
  care with different staffing and different vaccine allocations.
- **`Dawaki General Hospital` vs `Dawakin Kudu General Hospital`** — 20 metres
  apart, but *Dawakin Kudu* is a different Local Government Area from *Dawaki*.
- **`Jibga Fulani Health Post` vs `Jibga Health Post`** — 2.6 km apart, and
  `Fulani` may denote a distinct community's facility.

Each of these has a plausible, confident, wrong answer sitting right there. A
system optimising for match rate merges them all and looks excellent on a
dashboard. **In a Master Facility List a wrong merge isn't a lower score — it's
a clinic disappearing from the national list and losing its allocation.**

`review` is not the system failing. It is the system telling you where a human
is genuinely required — and there are far fewer of those than the raw residue
suggested.

## Step 8 — Every decision shows its working

A verdict you can't interrogate is not much use in a compliance setting. Each
edge carries the per-factor evidence and a content-addressed id.

In [8]:
import json
example = R[0]
print("A:", A[example["a_id"]]["name"])
print("B:", B[example["b_id"]]["name"])
print()
print(json.dumps(example, indent=2, default=str))

A: Gurduba Health Post
B: Gurduba Primary Health Care

{
  "a_id": 332,
  "b_id": 1174,
  "score": 0.6984,
  "decision": "review",
  "evidence": {
    "name": 0.837,
    "name_tftoken": 0.468,
    "geo": 0.882,
    "distance_km": 0.38
  },
  "distinctive_max": 0.837,
  "decision_id": "xwd:sha256:b5970ec22ca1965296ad78c38698bb87408a4939bb4e99b4e987e4d54b96aa43"
}


`decision_id` is a hash over the evidence **and** the exact representation
that produced it — engine version, thresholds, blocking strategy. Run this
notebook again tomorrow on the same data and you get the same id. That's what
makes a decision auditable six months later when someone asks why two clinics
became one.

## Step 9 — Export the review queue

The practical output of this notebook is a worklist for a human.

In [9]:
with open("review_queue.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["score", "distance_km", "osm_name", "grid3_name", "decision_id"])
    for m in R:
        w.writerow([f'{m["score"]:.4f}',
                    f'{m["evidence"].get("distance_km", 0):.2f}',
                    A[m["a_id"]]["name"], B[m["b_id"]]["name"], m["decision_id"]])

print(f"wrote review_queue.csv — {len(R):,} pairs for human adjudication")

wrote review_queue.csv — 289 pairs for human adjudication


## What you just did

1. Loaded two independent facility lists
2. Established that exact matching solves the easy majority and nothing else
3. Saw both failure modes of naive string similarity
4. Ran a calibrated crosswalk that strips type tokens and treats geography as
   support rather than proof
5. Split the result into *resolved*, *needs a human*, and *no candidate*
6. Inspected the refusals and found they were the right refusals
7. Exported a worklist with the evidence attached

**The takeaway:** most of a reconciliation is free, a chunk is automatable, and
a small remainder genuinely needs judgement. The value of a tool here is not
how much it matches — it's how honestly it draws that third line.

## Try it yourself

- Change `STATE` to `"Edo"` or `"Ondo"` (both are in `data/`)
- Swap OSM for `data/hfr_kano.csv` (the official registry via HDX) and compare
  the numbers — you should see a *higher* match rate, because that source
  shares data with GRID3 rather than being independent
- Look at the `no candidate` group: are those facilities OSM has and the
  government doesn't?

## Next

`02_same_person_across_documents.ipynb` — the same engine applied to people
rather than places, across real PDFs.